## Mental Health & Income Level: Data Exploration and Cleaning

This notebook explores and cleans two datasets — the IHME Global Burden of Disease 
data (prevalence and DALYs for 5 mental health disorders, 1990–2023, 204 countries) 
and the World Bank income classification — as the foundation for analyzing whether 
income level correlates with the burden of mental health disorders globally.

## Step 1: Load and Explore the IHME Dataset

First look at the raw data — checking shape, structure, and data types.

In [1]:
import pandas as pd
ihme = pd.read_csv(r"C:\Users\ASUS\Documents\Mental Health Dashboard\data\IHME-GBD_2023_DATA-666d879d-1\IHME-GBD_2023_DATA-666d879d-1.csv")
ihme.head()

,population_group_id,population_group_name,measure_id,measure_name,location_id,location_name,sex_id,sex_name,age_id,age_name,cause_id,cause_name,metric_id,metric_name,year,val,upper,lower
0,1,All Population,2,DALYs (Disability-Adjusted Life Years),8,Taiwan,3,Both,22,All ages,567,Depressive disorders,3,Rate,1990,273.491663,387.768598,178.123868
1,1,All Population,2,DALYs (Disability-Adjusted Life Years),8,Taiwan,3,Both,22,All ages,570,Bipolar disorder,3,Rate,1990,50.045391,74.111901,29.251486
2,1,All Population,2,DALYs (Disability-Adjusted Life Years),8,Taiwan,3,Both,22,All ages,571,Anxiety disorders,3,Rate,1990,318.227840,450.958814,204.408508
3,1,All Population,2,DALYs (Disability-Adjusted Life Years),8,Taiwan,3,Both,22,All ages,572,Eating disorders,3,Rate,1990,54.418122,86.743942,31.443198
4,1,All Population,2,DALYs (Disability-Adjusted Life Years),10,Cambodia,3,Both,22,All ages,559,Schizophrenia,3,Rate,1990,130.170157,169.307071,92.236136


In [2]:
print(ihme.shape)
ihme.info()

(69360, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69360 entries, 0 to 69359
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   population_group_id    69360 non-null  int64  
 1   population_group_name  69360 non-null  object 
 2   measure_id             69360 non-null  int64  
 3   measure_name           69360 non-null  object 
 4   location_id            69360 non-null  int64  
 5   location_name          69360 non-null  object 
 6   sex_id                 69360 non-null  int64  
 7   sex_name               69360 non-null  object 
 8   age_id                 69360 non-null  int64  
 9   age_name               69360 non-null  object 
 10  cause_id               69360 non-null  int64  
 11  cause_name             69360 non-null  object 
 12  metric_id              69360 non-null  int64  
 13  metric_name            69360 non-null  object 
 14  year                   69360 non-null  int

In [3]:
print(ihme['measure_name'].unique())
print(ihme['cause_name'].unique())
print(ihme['location_name'].nunique(), "unique countries")
print(ihme['year'].min(), "-", ihme['year'].max())

['DALYs (Disability-Adjusted Life Years)' 'Prevalence']
['Depressive disorders' 'Bipolar disorder' 'Anxiety disorders'
 'Eating disorders' 'Schizophrenia']
204 unique countries
1990 - 2023


In [4]:
ihme.isnull().sum()

population_group_id      0
population_group_name    0
measure_id               0
measure_name             0
location_id              0
location_name            0
sex_id                   0
sex_name                 0
age_id                   0
age_name                 0
cause_id                 0
cause_name               0
metric_id                0
metric_name              0
year                     0
val                      0
upper                    0
lower                    0
dtype: int64

In [5]:
ihme.duplicated().sum()

np.int64(0)

Initial checks show the IHME data has no missing values and no duplicate rows. 
Before dropping columns, checking which ones are actually constant across the 
entire dataset (and therefore carry no information).

In [5]:
for col in ['population_group_name', 'sex_name', 'age_name', 'metric_name']:
    print(col, ':', ihme[col].unique())

population_group_name : ['All Population']
sex_name : ['Both']
age_name : ['All ages']
metric_name : ['Rate']


Several columns aren't needed for this analysis — some only had one repeated value 
across all rows (like sex or age group), some were just ID codes for columns I already 
kept, and the upper/lower confidence bounds aren't needed since I'm using the main 
estimate (val). Dropping these keeps just the 5 columns I actually need: measure_name, 
location_name, cause_name, year, and val.

In [6]:
cols_to_drop = [ 'population_group_id','population_group_name','measure_id','location_id','sex_id','sex_name','age_id','age_name','cause_id','metric_id','metric_name','upper','lower']
ihme_clean = ihme.drop(columns=cols_to_drop)
ihme_clean.head()



,measure_name,location_name,cause_name,year,val
0,DALYs (Disability-Adjusted Life Years),Taiwan,Depressive disorders,1990,273.491663
1,DALYs (Disability-Adjusted Life Years),Taiwan,Bipolar disorder,1990,50.045391
2,DALYs (Disability-Adjusted Life Years),Taiwan,Anxiety disorders,1990,318.227840
3,DALYs (Disability-Adjusted Life Years),Taiwan,Eating disorders,1990,54.418122
4,DALYs (Disability-Adjusted Life Years),Cambodia,Schizophrenia,1990,130.170157


## Step 2: Load and Explore the World Bank Income Classification

Loading the World Bank's country income classification (FY2025), which will be 
merged onto the IHME data to group countries by income level.

In [7]:
wb = pd.read_excel(r"C:\Users\ASUS\Documents\Mental Health Dashboard\data\CLASS_2026_07_15.xlsx", sheet_name='List of economies')
wb.head()

,Economy,Code,Region,Income group,Lending category
0,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income,IDA
1,Albania,ALB,Europe & Central Asia,Upper middle income,IBRD
2,Algeria,DZA,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,IBRD
3,American Samoa,ASM,East Asia & Pacific,High income,NaN
4,Andorra,AND,Europe & Central Asia,High income,NaN


In [8]:
print(wb.shape)
wb.info()

(266, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Economy           265 non-null    object
 1   Code              265 non-null    object
 2   Region            218 non-null    object
 3   Income group      218 non-null    object
 4   Lending category  145 non-null    object
dtypes: object(5)
memory usage: 10.5+ KB


In [9]:
print(wb['Income group'].unique())
print(wb.isnull().sum())

['Low income' 'Upper middle income' 'High income' 'Lower middle income'
 nan]
Economy               1
Code                  1
Region               48
Income group         48
Lending category    121
dtype: int64


48 rows have no income group. Checking these before dropping them, to confirm 
they are aggregate/regional rows (e.g. "World", "East Asia & Pacific") rather 
than real countries.

In [10]:
wb[wb['Income group'].isnull()][['Economy','Code','Region']]

,Economy,Code,Region
218,NaN,NaN,NaN
219,Africa Eastern and Southern,AFE,NaN
220,Africa Western and Central,AFW,NaN
221,Arab World,ARB,NaN
222,Caribbean small states,CSS,NaN
223,Central Europe and the Baltics,CEB,NaN
224,Early-demographic dividend,EAR,NaN
225,East Asia & Pacific,EAS,NaN
226,East Asia & Pacific (excluding high income),EAP,NaN
227,East Asia & Pacific (IDA & IBRD),TEA,NaN


Confirmed — all 48 null rows are regional aggregates or summary categories, not 
real countries. Safe to exclude them, keeping only the 4 relevant columns.

In [11]:
wb_clean = wb.dropna(subset=['Income group'])[['Economy', 'Code', 'Region', 'Income group']]
print(wb_clean.shape)
wb_clean.head()

(218, 4)


,Economy,Code,Region,Income group
0,Afghanistan,AFG,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,Albania,ALB,Europe & Central Asia,Upper middle income
2,Algeria,DZA,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income
3,American Samoa,ASM,East Asia & Pacific,High income
4,Andorra,AND,Europe & Central Asia,High income


## Step 3: Match Country Names Between Datasets

Checking how many IHME country names match directly against the World Bank's 
naming convention before any cleanup.

In [12]:
ihme_countries = set(ihme_clean['location_name'].unique())
wb_countries = set(wb_clean['Economy'].unique())

matched = ihme_countries & wb_countries
unmatched = ihme_countries-wb_countries

print("Matched:",len(matched))
print("Unmatched",len(unmatched))
print(sorted(unmatched))

Matched: 172
Unmatched 32
['Bahamas', 'Bolivia (Plurinational State of)', 'Congo', 'Cook Islands', "Côte d'Ivoire", "Democratic People's Republic of Korea", 'Democratic Republic of the Congo', 'Egypt', 'Gambia', 'Iran (Islamic Republic of)', 'Kyrgyzstan', "Lao People's Democratic Republic", 'Micronesia (Federated States of)', 'Nauru', 'Niue', 'Palestine', 'Puerto Rico', 'Republic of Korea', 'Republic of Moldova', 'Saint Kitts and Nevis', 'Saint Lucia', 'Saint Vincent and the Grenadines', 'Sao Tome and Principe', 'Slovakia', 'Somalia', 'Taiwan', 'Tokelau', 'United Republic of Tanzania', 'United States Virgin Islands', 'United States of America', 'Venezuela (Bolivarian Republic of)', 'Yemen']


32 country names didn't match directly, mostly due to naming differences between 
the two sources. Searching the World Bank list using keywords from the 
unmatched names to find their likely equivalents.

In [13]:
wb_names = sorted(wb_clean['Economy'].unique())
keywords = ['Bahamas','Bolivia','Congo','Cook','Ivoire','Korea','Egypt','Gambia','Iran','Kyrgyz','Lao','Micronesia','Nauru','Niue','Palestin','Puerto','Moldova','Kitts','Lucia','Vincent','Tome','Slovak','Somalia','Taiwan','Tokelau','Tanzania','Virgin','United States','Venezuela','Yemen']

for n in wb_names:
    if any(k.lower() in n.lower() for k in keywords):
        print(n)

Bahamas, The
Bolivia
British Virgin Islands
Congo, Dem. Rep.
Congo, Rep.
Côte d’Ivoire
Egypt, Arab Rep.
Gambia, The
Iran, Islamic Rep.
Korea, Dem. People's Rep.
Korea, Rep.
Kyrgyz Republic
Lao PDR
Micronesia, Fed. Sts.
Moldova
Puerto Rico (U.S.)
Slovak Republic
Somalia, Fed. Rep.
St. Kitts and Nevis
St. Lucia
St. Vincent and the Grenadines
Taiwan, China
Tanzania
United States
Venezuela, RB
Virgin Islands (U.S.)
Yemen, Rep.


### Merging Income Classification

The IHME dataset uses UN-style full country names, while the World Bank file uses 
its own naming conventions (abbreviated or different official names). Before merging, 
country names in the IHME data are standardized to match the World Bank's naming, 
based on a manually verified mapping. A few small Pacific territories (Cook Islands, 
Nauru, Niue, Tokelau) are not classified by the World Bank and will not have an income 
group after the merge — this is expected, not a data error.


In [14]:
name_fix = {
    'Bahamas': 'Bahamas, The',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'United States Virgin Islands': 'Virgin Islands (U.S.)',
    'Congo': 'Congo, Rep.',
    'Democratic Republic of the Congo': 'Congo, Dem. Rep.',
    "Côte d'Ivoire": 'Côte d’Ivoire',
    'Egypt': 'Egypt, Arab Rep.',
    'Gambia': 'Gambia, The',
    'Iran (Islamic Republic of)': 'Iran, Islamic Rep.',
    "Democratic People's Republic of Korea": "Korea, Dem. People's Rep.",
    'Republic of Korea': 'Korea, Rep.',
    'Kyrgyzstan': 'Kyrgyz Republic',
    "Lao People's Democratic Republic": 'Lao PDR',
    'Micronesia (Federated States of)': 'Micronesia, Fed. Sts.',
    'Republic of Moldova': 'Moldova',
    'Puerto Rico': 'Puerto Rico (U.S.)',
    'Slovakia': 'Slovak Republic',
    'Somalia': 'Somalia, Fed. Rep.',
    'Saint Kitts and Nevis': 'St. Kitts and Nevis',
    'Saint Lucia': 'St. Lucia',
    'Saint Vincent and the Grenadines': 'St. Vincent and the Grenadines',
    'Taiwan': 'Taiwan, China',
    'United Republic of Tanzania': 'Tanzania',
    'United States of America': 'United States',
    'Venezuela (Bolivarian Republic of)': 'Venezuela, RB',
    'Yemen': 'Yemen, Rep.',
}

ihme_clean['location_name_wb'] = ihme_clean['location_name'].replace(name_fix)

In [15]:
matched_after = set(ihme_clean['location_name_wb'].unique()) & set(wb_clean['Economy'].unique())
unmatched_after = set(ihme_clean['location_name_wb'].unique()) - set(wb_clean['Economy'].unique())

print("Matched after fixing:", len(matched_after))
print("Still unmatched:", sorted(unmatched_after))

Matched after fixing: 198
Still unmatched: ['Cook Islands', 'Nauru', 'Niue', 'Palestine', 'Sao Tome and Principe', 'Tokelau']


In [16]:
wb_names = sorted(wb_clean['Economy'].unique())
for n in wb_names:
    if 'pal' in n.lower() or 'west bank' in n.lower() or 'gaza' in n.lower() or 'tome' in n.lower() or 'principe' in n.lower():
        print(n)

Nepal
Palau
West Bank and Gaza


In [17]:
for n in wb_names:
    if 'tom' in n.lower():
        print(repr(n))

'São Tomé and Príncipe'


In [18]:
name_fix.update({
    'Palestine': 'West Bank and Gaza',
    'Sao Tome and Principe': 'São Tomé and Príncipe',
})

ihme_clean['location_name_wb'] = ihme_clean['location_name'].replace(name_fix)

matched_final = set(ihme_clean['location_name_wb'].unique()) & set(wb_clean['Economy'].unique())
unmatched_final = set(ihme_clean['location_name_wb'].unique()) - set(wb_clean['Economy'].unique())

print("Matched:", len(matched_final))
print("Still unmatched:", sorted(unmatched_final))

Matched: 200
Still unmatched: ['Cook Islands', 'Nauru', 'Niue', 'Tokelau']


### Final Merge

200 of 204 IHME countries (98%) were successfully matched to a World Bank income 
classification. The remaining 4 (Cook Islands, Nauru, Niue, Tokelau) are small Pacific 
territories that the World Bank does not classify by income group — these rows will 
have a missing income_group value after the merge, which is expected.

In [19]:
wb_lookup = wb_clean.set_index('Economy')['Income group'].to_dict()

ihme_clean['income_group'] = ihme_clean['location_name_wb'].map(wb_lookup)

print(ihme_clean['income_group'].isnull().sum(), "rows with no income group")
ihme_clean.head()

1360 rows with no income group


,measure_name,location_name,cause_name,year,val,location_name_wb,income_group
0,DALYs (Disability-Adjusted Life Years),Taiwan,Depressive disorders,1990,273.491663,"Taiwan, China",High income
1,DALYs (Disability-Adjusted Life Years),Taiwan,Bipolar disorder,1990,50.045391,"Taiwan, China",High income
2,DALYs (Disability-Adjusted Life Years),Taiwan,Anxiety disorders,1990,318.227840,"Taiwan, China",High income
3,DALYs (Disability-Adjusted Life Years),Taiwan,Eating disorders,1990,54.418122,"Taiwan, China",High income
4,DALYs (Disability-Adjusted Life Years),Cambodia,Schizophrenia,1990,130.170157,Cambodia,Lower middle income


`location_name_wb` was only needed to match country names during the merge — 
dropping it now since it's no longer useful going forward.

In [20]:
ihme_clean = ihme_clean.drop(columns=['location_name_wb'])
ihme_clean.head()

,measure_name,location_name,cause_name,year,val,income_group
0,DALYs (Disability-Adjusted Life Years),Taiwan,Depressive disorders,1990,273.491663,High income
1,DALYs (Disability-Adjusted Life Years),Taiwan,Bipolar disorder,1990,50.045391,High income
2,DALYs (Disability-Adjusted Life Years),Taiwan,Anxiety disorders,1990,318.227840,High income
3,DALYs (Disability-Adjusted Life Years),Taiwan,Eating disorders,1990,54.418122,High income
4,DALYs (Disability-Adjusted Life Years),Cambodia,Schizophrenia,1990,130.170157,Lower middle income


Checking how many rows still have no income group after the merge, and confirming 
they belong only to the 4 expected unclassified territories.

In [21]:
print(ihme_clean['income_group'].isnull().sum(), "rows with no income group")
print(ihme_clean[ihme_clean['income_group'].isnull()]['location_name'].unique())

1360 rows with no income group
['Cook Islands' 'Tokelau' 'Niue' 'Nauru']


Confirming how many rows are usable for income-based analysis (i.e., have a 
non-null income_group) versus the total dataset.

In [22]:
total_rows = len(ihme_clean)
usable_rows = ihme_clean['income_group'].notna().sum()
excluded_rows = ihme_clean['income_group'].isna().sum()

print("Total rows:", total_rows)
print("Usable rows (has income group):", usable_rows)
print("Excluded rows (no income group):", excluded_rows)
print("Usable %:", round(usable_rows/total_rows*100, 2), "%")

Total rows: 69360
Usable rows (has income group): 68000
Excluded rows (no income group): 1360
Usable %: 98.04 %


Saving the cleaned and merged dataset as a checkpoint before moving into 
exploratory analysis.

In [23]:
ihme_clean.to_csv(r"C:\Users\ASUS\Documents\Mental Health Dashboard\data\ihme_income_merged.csv", index=False)